# Optimization Hyperparameters

## Batch size

Folk knowledge suggests setting batch sizes as powers of 2: $B = 16, 32, 64, ..., 512.$
Starting with $B = 32$ is recommended for image tasks {cite}`batch-size-32`.
Note that we may need to train with large batch sizes depending on the network architecture, the 
nature of the training distribution, or if we have large compute {cite}`imagenet1hour`.
Conversely, we may be forced to use small batches due to resource constraints with large models.

**Large batch.** 
Increasing $B$ with other parameters fixed can result in worse generalization ({numref}`02-large-batch-training`). This has been attributed to batch size decreasing gradient noise {cite}`learning-rate-function-of-batch-size`.
Intuitively, less sampling noise means that we can use a larger learning rate, since the loss surface is more stable to different samples. Indeed, {cite}`imagenet1hour` suggests scaling up the learning rate by the same factor that batch size is increased. {numref}`02-imagenet-1-hour` shows that the simple scaling rule works up to a certain point.


```{figure} ../../../img/nn/02-large-batch-training.png
---
name: 02-large-batch-training
width: 600px
align: center
---
{cite}`sharp_minima_bad` All models are trained in PyTorch using [Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html) 
with default parameters. Large batch training (LB) uses 10% of the dataset while small batch (SB) uses $B = 256$.
The table shows results for models that were not overfitted to the training distribution, with large batch size training having generally worse test performance.
```

**Small batch.** Using a small batch generally results in slow and unstable convergence since 
the loss surface is poorly approximated at each step. But may be necessary due to memory constraints.
Instead, a larger batch size can be simulated by accumulating gradients from "micro-batches" before performing a weight update. This is called **gradient accumulation**.
Here $\bar{B} \times S = B$, i.e. the actual batch size $\bar{B}$ times the number of gradient accumulation steps $S$ is equal to the ideal batch size $B.$ 

A simple implementation for batches with *fixed number of elements*[^1] is as follows:

```python
train_loader = DataLoader(dataset, batch_size=B//S, drop_last=True)

for i, (x, y) in enumerate(train_loader):
    outs = model(x)
    loss = loss_fn(outs, y) / S
    loss.backward()

    if (i + 1) % S == 0:
        optim.step()
        optim.zero_grad()
```

````{dropdown} ⓘ &nbsp; Batches of sequences of varying lengths
For batches consisting of sequences with varying lengths, the number of elements summed over in calculating the cross-entropy varies for each micro-batch. The correct approach here is to accumulate all contributions in a batch before normalizing, so that invariance with $S$ is satisfied:

```python
train_loader = DataLoader(dataset, batch_size=B//S, drop_last=True)

m = 0
for i, (x, y) in enumerate(train_loader):
    outs = model(x) # (B, C, T)
    loss = F.cross_entropy(outs, y, reduction="sum")
    loss.backward()
    m += (y != -100).int().flatten().sum()

    if (i + 1) % S == 0:
        for p in model.parameters():
            p.grad /= m
        optim.step()
        optim.zero_grad()
        m = 0
```
Here `-100` is the ignore index for `F.cross_entropy`.
````

Training takes longer by the same factor $S$. But observe that the backward and forward passes for each accumulation step are independent of each other (since the model does not update until the last step). Hence, we can improve training time by using multiple GPUs with [data parallelism](https://docs.nvidia.com/nemo-framework/user-guide/24.09/nemotoolkit/features/parallelisms.html#data-parallelism).

[^1]: The math trivially checks out whenever $B_a = \bar{B}$ for all $a$:
$\sum_{a=1}^S\sum_{i=1}^{B_a} \frac{1}{S \times B_a} \nabla \mathcal{L}_{ai}.$
But with output sequences, 
$\frac{1}{S}\sum_{a=1}^S \frac{1}{ N_a} \sum_{i=1}^{B_a} \sum_{t = 1}^{T_{ai}} \nabla \mathcal{L}_{ait}$
such that $N_a = \sum_{i=1}^{B_a} T_{ai}$ is the sum of sequence lengths which generally varies (e.g. ignoring padding tokens).
This basically separates the entire fraction into partial fractions. <br><br>Recall that cross-entropy for a batch of sequences is calculated by accumulating all instance losses regardless of which sequence it belongs in. Hence, the invariance for any $S = 1, 2, \ldots$ is violated. Invariance is satisfied with: $\frac{1}{N} \sum_{a=1}^S \sum_{i=1}^{B_a} \sum_{t = 1}^{T_{ai}} \nabla \mathcal{L}_{ait}$ where $N = \sum_{a=1}^S \sum_{i=1}^{B_a} T_{ai}.$ That is, normalization is applied only after adding all the terms. Finally, this example demonstrates how silent calculation errors can cause significant differences in training outcomes and cumulative training cost in deep learning. See this [blog post](https://unsloth.ai/blog/gradient).

In [ ]:
import torch
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim.optimizer
from torch.utils.data import TensorDataset, DataLoader

N = 7331
T = 20
C = 10

def reseed():
    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)

def get_model():
    reseed()
    return torch.nn.Sequential(
        nn.Linear(4, 5),
        nn.ReLU(),
        nn.Linear(5, 10)
    )

def get_dataloader(B, masked=False):
    reseed()
    mask = torch.randint(0, 2, size=(N, T))
    x = torch.randn(size=(N, T, 4))
    y = torch.randint(0, C, size=(N, T))
    dataset = TensorDataset(
        x,
        y * mask + 100 * (mask - 1) if masked else y
    )
    return DataLoader(dataset, batch_size=B, drop_last=True)

for S in [1, 2, 4, 8]:
    reseed()
    B = 32
    model = get_model()
    optim = torch.optim.SGD(model.parameters(), lr=0.01)
    train_loader = get_dataloader(B // S, masked=False)
    
    m = 0
    for i, (x, y) in enumerate(train_loader):
        outs = model(x).permute(0, 2, 1)
        loss = F.cross_entropy(outs, y, reduction="sum")
        loss.backward()
        m += (y != -100).int().flatten().sum()

        if (i + 1) % S == 0:
            for p in model.parameters():
                p.grad /= m
            optim.step()
            optim.zero_grad()
            m = 0

    xt = torch.randn(size=(256, T, 4))
    yt = torch.randint(0, 9, size=(256, T))
    print(F.cross_entropy(model(xt).permute(0, 2, 1), yt).item())

<br>

**Remark.** GPU is underutilized when $B$ is small, and we can get OOM when $B$ is large.
In general, hardware constraints should be considered in parallel with theory.
GPU can idle if there is lots of CPU processing on a large batch, for example. One can set 
`pin_device=True` can be set in the data loader to speed up data transfers to 
the GPU by leveraging [page locked memory](https://leimao.github.io/blog/Page-Locked-Host-Memory-Data-Transfer/). 
Similar tricks
({numref}`02-gpu-tricks`) have to be tested empirically to see whether it works on your
use-case. These are hard to figure out based on first principles. 

```{figure} ../../../img/nn/02-gpu-tricks.png
---
name: 02-gpu-tricks
width: 600px
align: center
---
A [tweet](https://twitter.com/karpathy/status/1299921324333170689?s=20) by Andrej Karpathy
on tricks to optimize Pytorch code. The linked [video](https://www.youtube.com/watch?v=9mS1fIYj1So).
```

## Learning rate

Finding an optimal learning rate (LR) is essential for test performance and 
faster convergence, even for adaptive optimizers like Adam. In practice, we first determine the batch size based on hardware constraints such as GPU/CPU efficiency, memory, and data transfer latency. Then, we tune the learning rate by (1) selecting a **base LR** and (2) an LR decay **policy** or **schedule**. If we find a good base LR, and want to change the batch size, we have to scale the LR linearly with the same factor {cite}`imagenet1hour` {cite}`learning-rate-function-of-batch-size`. 
This means smaller LR for smaller batches, and vice-versa ({numref}`02-imagenet-1-hour`).

```{figure} ../../../img/nn/02-imagenet-1-hour.png
---
name: 02-imagenet-1-hour
width: 500px
align: center
---
From {cite}`imagenet1hour`.  For all experiments 
$B \leftarrow aB$ and $\eta \leftarrow a\eta$
sizes are set. Note that a simple warmup phase for the first few epochs of
training until the learning rate stabilizes to $\eta$ since 
early steps are away from any minima, hence can be unstable. 
All other hyper-parameters are kept fixed. Using this
simple approach, accuracy of our models is invariant to minibatch
size (up to an 8k minibatch size). The authors were able to train
ResNet-50 on ImageNet in 1 hour using 256 GPUs with 90% scaling efficiency relative to the baseline of using 8 GPUs.
```

**LR finder.** The following is a parameter-free approach to finding a good base learning rate.
The idea is to select a base learning rate that is as large as possible without the loss diverging
at early steps of training.
This allows the optimizer to initially explore the surface with less risk of 
getting stuck in plateaus.

In [ ]:
from chapter import *

In [ ]:
num_steps = 1000
lre_min = -2.0
lre_max =  0.6
lre = torch.linspace(lre_min, lre_max, num_steps)
lrs = 10 ** lre
w = nn.Parameter(torch.FloatTensor([-4.0, -4.0]), requires_grad=True)
optim = Adam([w], lr=lrs[0])

losses = []
for k in range(num_steps):
    optim.lr = lrs[k]   # (!) change LR at each step
    optim.zero_grad()
    loss = pathological_loss(w[0], w[1])
    loss.backward()
    optim.step()
    losses.append(loss.item())

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(lrs.detach(), losses)
plt.xlabel("learning rate")
plt.ylabel("loss")
plt.grid(linestyle='dotted')
plt.axvline(2.5, color='k', linestyle='dashed', label='base LR')
plt.legend();

Notice that sampling is biased towards small learning rates. This makes sense since large learning rates tend to diverge. The graph is not representative for practical problems since the network is small and the loss surface is relatively simple. But following the algorithm, `lr=2.0` may be chosen as the base learning rate.

<br>

**LR scheduling.** Learning rate has to be decayed in some way help with convergence. 
Recall that this happens automatically using adaptive methods, but having a loss surface independent 
policy still helps, especially when given a predetermined computational budget.
The following modifies the training script to include a simple schedule. Repeating the same experiment above for RMSProp and GD which had issues with oscillation:

In [ ]:
def train_curve(
    optim: OptimizerBase, 
    optim_params: dict, 
    w_init=[5.0, 5.0], 
    loss_fn=pathological_loss, 
    num_steps=100
):
    """Return trajectory of optimizer through loss surface from init point."""

    w_init = torch.tensor(w_init).float()
    w = nn.Parameter(w_init, requires_grad=True)
    optim = optim([w], **optim_params)
    points = [torch.tensor([w[0], w[1], loss_fn(w[0], w[1])])]
    
    for step in range(num_steps):
        optim.zero_grad()
        loss = loss_fn(w[0], w[1])
        loss.backward()
        optim.step()

        # logging
        with torch.no_grad():
            z = loss.unsqueeze(dim=0)
            points.append(torch.cat([w.data, z]))

        # LR schedule (!)
        if step % 70 == 0:
            optim.lr *= 0.5

    return torch.stack(points, dim=0).numpy()

In [ ]:
import chapter
chapter.train_curve = train_curve

In [ ]:
label_map_gdm = {"lr": r"$\eta$", "momentum": r"$\beta$"}
label_map_rmsprop = {"lr": r"$\eta$", "beta": r"$\beta$"}
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
plot_gd_steps(ax, optim=GD,      optim_params={"lr": 3.0},              w_init=[-5.0, 5.0], label_map=label_map_gdm,     color="red")
plot_gd_steps(ax, optim=RMSProp, optim_params={"lr": 3.0, "beta": 0.9}, w_init=[-5.0, 5.0], label_map=label_map_rmsprop, color="black")

ax[0].set_xlim(-6, 6)
ax[0].set_ylim(-8, 6)
ax[1].set_xlabel("steps")
ax[1].set_ylabel("loss")
ax[1].axvline(70, linestyle='dashed')
ax[1].axvline(140, linestyle='dashed')
ax[1].axvline(210, linestyle='dashed', label='LR step', zorder=1)
ax[1].grid(linestyle="dotted", alpha=0.8)
ax[0].legend(fontsize=9)
ax[1].legend(fontsize=9);

Learning rate decay decreases GD oscillation drastically. The schedule $\boldsymbol{\boldsymbol{\Theta}}^{t+1} = \boldsymbol{\boldsymbol{\Theta}}^{t} - \eta \frac{1}{\alpha^t} \, \boldsymbol{\mathsf{m}}^{t}$ where $\alpha^t = 2^{\lfloor t / 100 \rfloor}$ is known as **step LR decay**. Note that this augments the second-moment for RMSProp which already auto-tunes the learning rate. Here we are able to start with a large learning rate allowing the optimizer to escape the first plateau earlier than before. Note that decay only decreases learning rate which can cause slow convergence. Some schedules implement **warm restarts** to fix this ({numref}`02-sgd-warm-restarts`).

**Remark.** For more examples of learning rate decay schedules [see here](https://d2l.ai/chapter_optimization/lr-scheduler.html#schedulers)  (e.g. **warmup** which initially gradually increases learning rate
since SGD at initialization can be unstable with large LR). Also see [PyTorch docs](https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate) on LR schedulers implemented in the library. For example, the schedule **reduce LR on plateau** which reduces the learning rate when a metric has stopped improving is implemented in PyTorch as `ReduceLROnPlateau` in the `torch.optim.lr_scheduler` library.

```python
# Example: PyTorch code for chaining LR schedulers
optim = SGD(model.parameters(), lr=0.01, momentum=0.9)
scheduler1 = ExponentialLR(optim, gamma=0.9)
scheduler2 = MultiStepLR(optim, milestones=[30,80], gamma=0.1)

for epoch in range(10):
    for x, y in dataset:
        optim.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optim.step()
    
    # LR step called after optimizer update! ⚠⚠⚠
    scheduler1.step()
    scheduler2.step()
```

<br>

```{figure} ../../../img/nn/02-sgd-warm-restarts.png
---
name: 02-sgd-warm-restarts
width: 60%
align: center
---
Cosine annealing starts with a large learning rate that is relatively rapidly decreased to a minimum value before being increased rapidly again. This resetting acts like a simulated restart of the model training and the re-use of good weights as the starting point of the restart is referred to as a "warm restart" in contrast to a "cold restart" at initialization. Source: {cite}`sgd-warm-restarts`
```

```{figure} ../../../img/nn/02-snapshot-ensembles.png
---
name: 02-snapshot-ensembles
width: 80%
align: center
---
Effect of cyclical learning rates. Each model checkpoint for each LR warm restart (which often correspond to a minimum) can be used to create an ensemble model. Source: {cite}`snapshot-ensembles`
```

## Momentum

Good starting values for SGD momentum are $\beta = 0.9$ or $0.99$. Adam is easier to use out of the box where we like to keep the default parameters. If we have resources, and we want to push test performance, we can
tune SGD which is known to generalize better than Adam with more epochs. See {cite}`sgd_better_than_adam` where it is shown that Adam is more stable at sharp minima which tend to generalize worse than flat ones ({numref}`02-sharp-optim`).

```{figure} ../../../img/nn/02-sharp-optim.png
---
name: 02-sharp-optim
width: 80%
align: center
---
A conceptual sketch of flat and sharp minima. The Y-axis indicates value of the loss
function and the X-axis the variables. Source: {cite}`sharp_minima_bad`
```

**Remark.** In principle, optimization hyperparameters affect training 
and not generalization. But the situation is more complex with SGD, where stochasticity
contributes to regularization. This was shown above where choice of batch size influences
the 
generalization gap. Also
recall that for batch GD (i.e. $B = N$ in SGD), consecutive gradients approaching a minimum 
roughly have the same direction. 
This should not happen with SGD with $B \ll N$ in the learning regime as different samples 
will capture different aspects of the loss surface.
Otherwise, the network is starting to overfit. So in practice, optimization hyperparameters 
are tuned on the validation set as well.